In [1]:
# Ячейка 1: установка
!pip install -q lightgbm catboost xgboost optuna pyarrow

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score
import gc, os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 18.1 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Тогда данные в /content/drive/MyDrive/alfa_credit/
DATA_DIR = '/content/drive/MyDrive/alfa_credit'

Mounted at /content/drive


In [5]:
# Ячейка 3: загрузка
train = pd.read_parquet(f'{DATA_DIR}/train_dataset_small.pq')
test  = pd.read_parquet(f'{DATA_DIR}/test_dataset_small.pq')
feats = pd.read_parquet(f'{DATA_DIR}/features_small.pq')
desc  = pd.read_csv(f'{DATA_DIR}/feature_description.csv')

print("train:", train.shape, train.columns.tolist())
print("test:", test.shape, test.columns.tolist())
print("feats:", feats.shape)
print("desc:")
print(desc.head(20))

print("\nЦелевая переменная (если есть target колонка):")
target_col = [c for c in train.columns if 'target' in c.lower() or c == 'y']
print(target_col)
print(train[target_col[0]].value_counts() if target_col else "ищи таргет глазами")

train: (845110, 23) ['app_id', 'request_received', 'date_part', 'request_id', 'req_loan_amount', 'req_term', 'verif_compl', 'offer_id', 'variant_no', 'offer_type', 'pil1mtrx_offer', 'is_deal', 'rate', 'term', 'limit', 'risk_level_map', 'eva', 'eva_perc', 'verif_need', 'need_2ndfl', 'ncl', 'channel', 'basket_name']
test: (936883, 22) ['app_id', 'request_received', 'date_part', 'request_id', 'req_loan_amount', 'req_term', 'verif_compl', 'offer_id', 'variant_no', 'offer_type', 'pil1mtrx_offer', 'rate', 'term', 'limit', 'risk_level_map', 'eva', 'eva_perc', 'verif_need', 'need_2ndfl', 'ncl', 'channel', 'basket_name']
feats: (76583, 280)
desc:
         Обозначение                 Сет  \
0             app_id                  ID   
1   request_received                  ID   
2          date_part                  ID   
3         request_id                  ID   
4           offer_id                  ID   
5    req_loan_amount  Признаки из заявки   
6           req_term  Признаки из заявки   
7 

In [6]:
# Ячейка 4: соберём датасет и обучим LightGBM lambdarank
# Названия колонок придётся подставить по факту после п.4

KEY_CLIENT = 'client_id'        # подставь реальное имя
KEY_REQ    = 'request_id'
KEY_VAR    = 'variant_no'
TARGET     = 'target'            # подставь реальное имя

# Джойн: заявка + фичи клиента
train_full = train.merge(feats, on=KEY_CLIENT, how='left')
test_full  = test.merge(feats,  on=KEY_CLIENT, how='left')

# Сортировка по request_id обязательна для group-based ranker'а
train_full = train_full.sort_values(KEY_REQ).reset_index(drop=True)

feature_cols = [c for c in train_full.columns
                if c not in [KEY_REQ, KEY_CLIENT, TARGET]]

# GroupKFold по request_id
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(train_full))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        train_full, train_full[TARGET], groups=train_full[KEY_REQ])):

    tr = train_full.iloc[tr_idx]
    va = train_full.iloc[va_idx]

    tr_groups = tr.groupby(KEY_REQ).size().values
    va_groups = va.groupby(KEY_REQ).size().values

    dtr = lgb.Dataset(tr[feature_cols], tr[TARGET], group=tr_groups,
                       categorical_feature=[KEY_VAR])
    dva = lgb.Dataset(va[feature_cols], va[TARGET], group=va_groups,
                       categorical_feature=[KEY_VAR], reference=dtr)

    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'eval_at': [5],
        'lambdarank_truncation_level': 5,
        'learning_rate': 0.05,
        'num_leaves': 127,
        'min_data_in_leaf': 100,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
    }

    model = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(100)])

    oof[va_idx] = model.predict(va[feature_cols])
    print(f"Fold {fold} done")

KeyError: 'client_id'